## Instructions

Run the first cell. 
- If it does not run you probably are not in your `uv` environment. make sure to start `jupyter lab` by running
```
uv run --with jupyter jupyter lab
```
from a terminal with your `uv` environment active. If a kernal is not selected, then `Python 3 (ipykernel)` is probably the right choice 

In [ ]:
from cds_dashboard.cds_api_utils.Query import QueryCosmicDSApi
import pandas as pd
import matplotlib.pyplot as plt
import astropy.constants as const
import astropy.units as u
from ipywidgets import interact
import ipywidgets as widgets

q = QueryCosmicDSApi()
def get_slope(x, y):
    # slope through origin
    return nan if (sum(x**2) == 0) else (sum(x * y) / sum(x**2))

def slope2age(h0):
    return (1 / (h0 * u.km / u.s / u.Mpc)).to(u.Gyr).value

def get_summary(df):
    rows = []
    for student_id, g in df.groupby("student_id"):
        # has all data
        if len(g) != 5: continue
    
        # get H0
        H0 = get_slope(g["est_dist_value"].to_numpy(), g["velocity_value"].to_numpy())
    
        # table is id, h0, age
        rows.append({
            "student_id": student_id,
            "H0_km_s_Mpc": H0,
            "age_Gyr": slope2age(H0),
        })
    
    return pd.DataFrame(rows)

def plot(class_id, return_df = True):
    m = q.get_class_data(class_id=class_id)
    if m is not None and len(m) > 0:
        df = pd.DataFrame(m)
        clean = df.dropna(subset=["est_dist_value", "velocity_value"])
        summary = get_summary(clean)
        
        fig, axs = plt.subplots(1, 2, figsize=(9, 4))
        
        # colored by student_id
        axs[0].scatter(df['est_dist_value'], df['velocity_value'], c = df['student_id'], cmap='jet', edgecolors='k')
        axs[0].set_xlabel('Distance (Mpc)')
        axs[0].set_ylabel('Velocity (km/s)')
    
        
        
        # rwdith makes the bars 90% the width of the bin for ~~style~~ (default is 1)
        axs[1].hist(summary['age_Gyr'], histtype='bar', rwidth=.9)
        axs[1].set_xlabel('Age (Gyr)')

        fig.suptitle(f"Data for class {class_id}")
        if return_df:
            return df, summary
    else:
        print(f"No data for {class_id}")
    

In [ ]:
# Either Plot one at at ime
meas, summary = plot(194)

In [ ]:
# Or try interactive (it has a text box to enter the class_id). can be wonky
interact(lambda class_id: plot(class_id, False), class_id=widgets.IntText(value = 337))